# What this dataset is, and what it does and does not support

Everything here reads committed CSVs (`data/csv/`, `experiments/csv/`) — no model is fitted, no
network call is made, and nothing is written. Every number is traceable to a run recorded in
`experiments/*/`, each carrying a data manifest hash, a git SHA and a seed.

Run order matters: section 3 (the target's tails) and section 7 (multiple comparisons) are the two
that most change how the rest should be read.

## 1. Setup

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 140); pd.set_option('display.max_columns', 50)

# The panel is read from the committed parquet, not from a CSV copy. A derived CSV in git is a
# second source of truth that drifts; data/ is gitignored precisely so that cannot happen.
panel = pd.read_parquet('../data/training_set.parquet')

sites = (panel.groupby(['cluster','site_id'])
         .agg(n_obs=('obs_date','size'), first=('obs_date','min'), last=('obs_date','max'),
              lat=('latitude','first'), lon=('longitude','first'), elev=('elevation','first'),
              fz_mean=('forward_z','mean'), fz_sd=('forward_z','std'))
         .round(4).reset_index())

# Experiment results DO ship as CSV: they are the record of runs, not a copy of the input data.
folds = pd.read_csv('../experiments/csv/fold_results.csv')
cells = pd.read_csv('../experiments/csv/cell_summary.csv')
print(f'panel {panel.shape} | sites {sites.shape} | folds {folds.shape} | cells {cells.shape}')

## 2. The unit of analysis

The headline `n` is 4,596 rows. That is not the sample size for anything spatial: the independent
units are **4 clusters**, and monthly observations of one site are strongly autocorrelated. Any
leave-one-cluster-out result is an average of four numbers.

In [ ]:
print(panel.groupby('cluster').agg(rows=('site_id','size'), sites=('site_id','nunique'),
                                   obs_per_site=('site_id', lambda s: round(len(s)/s.nunique(),1)),
                                   lat=('latitude','mean'), elev=('elevation','mean')).round(1))
print(f"\nindependent spatial units: {panel.cluster.nunique()}  (this is the n that governs a transferability claim)")

### Sites are lattice points, not farms

`docs/RESEARCH_SUMMARY.md` states this plainly and it constrains every claim downstream: no row
here is a real farm, so nothing measured is an agronomic outcome.

## 3. The target, and a tail problem nobody has addressed

`forward_z` is a peer-standardised z-score, so it should be roughly N(0,1). It is not.

In [ ]:
fz = panel.forward_z
print(f'mean {fz.mean():+.4f}   sd {fz.std():.4f}   min {fz.min():+.2f}   max {fz.max():+.2f}')
print(f'|z| > 3 : {(fz.abs()>3).sum()} rows ({(fz.abs()>3).mean():.2%})')
print(f'|z| > 5 : {(fz.abs()>5).sum()} rows')
print(f'share <= -1.0 (the tau defining an event): {(fz<=-1).mean():.1%}')

fig, ax = plt.subplots(1, 2, figsize=(11,3.5))
ax[0].hist(fz, bins=120); ax[0].set_title('forward_z'); ax[0].axvline(-1, color='r', ls='--')
ax[1].hist(fz.clip(-4,4), bins=80); ax[1].set_title('clipped to +/-4'); ax[1].axvline(-1, color='r', ls='--')
plt.tight_layout()

An 11-sigma value in a quantity that is by construction a z-score means the standardisation is
dividing by a very small cohort standard deviation. The peer cohort requires only 5 members, so a
tight cohort produces enormous z. **These tails will dominate any squared-error fit**, and no
experiment in this repository has yet tested sensitivity to them. Worth checking before the
estimand is trusted further.

In [ ]:
# Which cohorts produce the extremes? Small cohorts, or genuinely anomalous fields?
ext = panel.loc[panel.forward_z.abs() > 5, ['cluster','site_id','obs_date','forward_z','ndvi','ndvi_z_peer']]
print(ext.sort_values('forward_z').to_string(index=False))

## 4. E01 — the field effect that started this

A per-field fixed effect accounts for **34.5%** of `forward_z` variance, CI [0.236, 0.435]
(`experiments/E01_variance_decomposition.out`, reproduced by `argotech.lab.variance`). The
climatology baseline that beat the incumbent model *is* an estimator of that effect.

In [ ]:
# Between-site spread of the per-site mean, straight from sites.csv
print(f"between-site sd of site means : {sites.fz_mean.std():.4f}")
print(f"mean within-site sd            : {sites.fz_sd.mean():.4f}")
sites.fz_mean.hist(bins=40, figsize=(6,3)); plt.title('per-site mean forward_z (the field effect)'); plt.axvline(0, color='r', ls='--');

## 5. The peer reference — coverage is the thing band width actually buys

In [ ]:
cov = (folds[folds.split=='spatial']
       .dropna(subset=['peer_coverage'])
       .groupby(['peer_key','lat_band','elev_band'])['peer_coverage'].mean().reset_index())
print(cov.round(3).to_string(index=False))

`cluster_month` reports 0.00 coverage on every spatial fold. That is not a bug and not poor
performance: the bucket key is the cluster's own identity, so a held-out cluster has no bucket at
all. The feature cannot be formed, the target cannot be built, and every fold is skipped.

## 6. Results — spatial versus temporal, always with the interval

In [ ]:
def table(target, split):
    t = cells[(cells.target==target) & (cells.split==split) & (cells.experiment=='E02')]
    return t[['peer_key','arm','net_benefit_mean','ci_lo','ci_hi','ci_excludes_zero','folds_scored']] \
            .sort_values('net_benefit_mean', ascending=False).round(4)

print('level_z / spatial  (the only peer_key-comparable target)'); print(table('level_z','spatial').to_string(index=False))
print('\nlevel_z / temporal'); print(table('level_z','temporal').to_string(index=False))

## 7. The multiple-comparisons check that governs everything above

Many cells were tested. Some will clear a 95% interval by chance. This is the number that decides
how much of section 6 to believe.

In [ ]:
c = cells.dropna(subset=['ci_excludes_zero'])
for split in ('spatial','temporal'):
    s = c[c.split==split]
    pos = s[(s.ci_excludes_zero==True) & (s.ci_lo>0)]
    chance = 0.025*len(s)   # two-sided 95%: ~2.5% land significant-positive by luck
    print(f'{split:9s}: {len(s):4d} tested, {len(pos):3d} significant-positive, ~{chance:.1f} expected by chance -> {len(pos)/chance:.1f}x')

**Spatial sits within a factor of two of pure noise; temporal is roughly eight times chance.**
No correction has been applied. The spatial hits are also internally incoherent — `level_z`/
`geo_month`/boosted clears zero at 500 m and 2000 m but not at the 1000 m sitting between them,
a pattern with no mechanism.

Defensible reading: **temporal skill is established; spatial skill is not distinguishable from
multiple-testing noise at four independent spatial units.**

## 8. The replication

`within_xy` / `geo_month` / spatial / linear at +0.0404 was the strongest single result. Its
per-fold decomposition is why it does not survive.

In [ ]:
r = folds[(folds.experiment=='E03-replication') & (folds.split=='spatial') & (folds.arm.isin(['linear','boosted']))]
print(r.pivot_table(index='fold', columns='arm', values=['net_benefit','event_rate','n']).round(4).to_string())
print('\nmax attainable net benefit IS the event rate, so folds are not on a common scale.')
print('The fold carrying the positive mean is also the one with the highest ceiling.')

## 9. What to do next

1. **The tails in section 3.** Untested sensitivity in the target itself.
2. **More clusters.** Four independent units is the binding constraint on every spatial claim;
   no amount of modelling fixes it.
3. **Real farms.** `backend_schema.list_fields` already returns registered farms with coordinates;
   nothing here is a farm.
4. **Pre-register the next comparison.** Section 7 is what testing 165 cells without a correction
   costs.